# Set WD to Root

In [1]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

print(Path.cwd())

d:\1_Document\A-02_Notebooks\1_UWA_Learning_Archive\02_Units\Y1S2_CITS4012 - Natural Language Processing\Assignment\Assignment_02\repo


In [2]:
from pathlib import Path
import sentencepiece as spm
from scripts.data_loader.piqa_dataloader import tokenize_piqa_dataframe

In [3]:
BASE_PATH = Path('./datasets/PIQA')
BASE_PATH = BASE_PATH.resolve()

# Test Batch Tokenize

In [4]:
# ============= Load Tokenizer ============================
tokenizer = spm.SentencePieceProcessor(
    model_file="./scripts/tokenizer/piqa_bpe_v3.model"
)

In [6]:
train_dataset, valid_dataset, test_dataset = tokenize_piqa_dataframe(tokenizer=tokenizer, base_path=BASE_PATH)

In [8]:
train_dataset.head()

,goal,sol1,sol2,label,goal_tokens,sol1_tokens,sol2_tokens
0,"When boiling butter, when it's ready, you can",Pour it onto a plate,Pour it into a jar,1,"[991, 1490, 597, 7915, 561, 313, 7935, 7900, 1...","[981, 313, 766, 263, 1330]","[981, 313, 368, 263, 867]"
2,how do you indent something?,leave a space before starting the writing,press the spacebar,0,"[451, 357, 292, 7127, 1143, 7922]","[1256, 263, 1854, 633, 2460, 266, 4665]","[753, 266, 1854, 3910]"
4,Clean tires,"Pour water, cape off caked on dirt. Use speed...","Pour water, scrape off caked on dirt. Use a st...",1,"[1437, 3941]","[981, 361, 7915, 269, 553, 485, 269, 2263, 306...","[981, 361, 7915, 2257, 485, 269, 2263, 306, 11..."
5,how do you taste something?,smell it enough to taste it.,place it in your mouth to taste.,1,"[451, 357, 292, 1830, 1143, 7922]","[1639, 313, 1152, 278, 1830, 313, 7913]","[458, 313, 286, 324, 1610, 278, 1830, 7913]"
6,"To create a makeshift ice pack,",take a sponge and soak it in oil. Put the spon...,take a sponge and soak it in water. Put the sp...,1,"[355, 1192, 263, 6489, 737, 1427, 7915]","[735, 263, 1565, 282, 1220, 313, 286, 575, 791...","[735, 263, 1565, 282, 1220, 313, 286, 361, 791..."


# Test Load DataLoaders

In [ ]:
from scripts.data_loader.piqa_dataloader import get_piqa_dataloaders
from pathlib import Path
import sentencepiece as spm

BASE_PATH = Path('./datasets/PIQA')
BASE_PATH = BASE_PATH.resolve()

tokenizer = spm.SentencePieceProcessor(
    model_file="./scripts/tokenizer/piqa_bpe_v3.model"
)


In [6]:
train_loader_bucket, validate_loader, test_loader = get_piqa_dataloaders(base_path=BASE_PATH, tokenizer=tokenizer, batch_size=1000)
train_loader_non_bucket, validate_loader, test_loader = get_piqa_dataloaders(base_path=BASE_PATH, tokenizer=tokenizer, batch_size=1000, use_bucket=False)

In [9]:
print(type(train_loader.dataset))
print(train_loader.dataset)

<class 'scripts.data_loader.piqa_dataloader.PIQADataset'>


# 测试分桶

In [7]:
total_tokens = 0
valid_tokens = 0

for input_1, input_2, mask_1, mask_2, labels in train_loader_bucket:
    total_tokens += input_1.numel() + input_2.numel()
    valid_tokens += mask_1.sum().item() + mask_2.sum().item()

padding_ratio = 1 - valid_tokens / total_tokens

print(f"Padding ratio (use bucket): {padding_ratio:.2%}")


total_tokens = 0
valid_tokens = 0

for input_1, input_2, mask_1, mask_2, labels in train_loader_non_bucket:
    total_tokens += input_1.numel() + input_2.numel()
    valid_tokens += mask_1.sum().item() + mask_2.sum().item()

padding_ratio = 1 - valid_tokens / total_tokens

print(f"Padding ratio (not use bucket): {padding_ratio:.2%}")


Padding ratio (use bucket): 34.17%
Padding ratio (not use bucket): 86.39%


# 句子复现测试

In [8]:
input_1, input_2, mask_1, mask_2, labels = next(iter(train_loader_bucket))

text_1 = tokenizer.decode_ids(input_1[0][mask_1[0]].tolist())
text_2 = tokenizer.decode_ids(input_2[0][mask_2[0]].tolist())

print(text_1)
print(text_2)

[CLS] How do I clean super filthy sneakers that I have?[SEP] Remove dirt or debris from the shoes. Use an old toothbrush or a small shoe brush and gently stroke the dirt or debris that is adhered to the shoe. ... Clean the soles using a flour paste. ... Pretreat any stains with a stain remover. ... Wash in a washing machine on the gentle cycle. ... Air dry the shoes.
[CLS] How do I clean super filthy sneakers that I have?[SEP] Remove dirt or debris from the shoes. Use an old toothbrush or a small shoe brush and gently stroke the dirt or debris that is adhered to the shoe. ... Clean the soles using a baking soda paste. ... Pretreat any stains with a stain remover. ... Wash in a washing machine on the gentle cycle. ... Air dry the shoes.


# 测试关闭 Goal

In [10]:
train_loader_no_goal, validate_loader, test_loader = get_piqa_dataloaders(base_path=BASE_PATH, tokenizer=tokenizer, batch_size=1000, include_goal=False)

In [13]:
input_1, input_2, mask_1, mask_2, labels = next(iter(train_loader_no_goal))

text_1 = tokenizer.decode_ids(input_1[0][mask_1[0]].tolist())
text_2 = tokenizer.decode_ids(input_2[0][mask_2[0]].tolist())

print(text_1)
print(text_2)

[CLS] dry your hands off with paper towels instead of hand dryer.
[CLS] dry your hands off using the hand dryer and then rinse them.
